## Importation des modules

In [1]:
import matplotlib as plt
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Lire le fichiers d'entrainement

In [2]:
pd_Data = pd.read_csv("heart_2020_uncleaned.csv")

## J'ai créé une ligne facultative pour créer un fichier de test du pipeline

In [3]:
pd_pipeline = pd_Data.drop('HeartDisease', axis=1)

pd_pipeline.to_csv("heart_topredic.csv", index=False)

### afficher les infos des données

In [4]:
pd_Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319795 entries, 0 to 319794
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   HeartDisease      319795 non-null  object 
 1   BMI               319794 non-null  float64
 2   Smoking           319795 non-null  object 
 3   AlcoholDrinking   319794 non-null  object 
 4   Stroke            319795 non-null  object 
 5   PhysicalHealth    319793 non-null  float64
 6   MentalHealth      319795 non-null  float64
 7   DiffWalking       319795 non-null  object 
 8   Sex               319794 non-null  object 
 9   AgeCategory       319794 non-null  object 
 10  Race              319794 non-null  object 
 11  Diabetic          319794 non-null  object 
 12  PhysicalActivity  319795 non-null  object 
 13  GenHealth         319794 non-null  object 
 14  SleepTime         319794 non-null  float64
 15  Asthma            319795 non-null  object 
 16  KidneyDisease     31

- HeartDisease = Bool
- BMI = Float
- Smoking = Bool
- AlcoholDrinking = Bool
- Stroke = Bool
- PhysicalHealt = Float
- MentalHealth = Float
- DiffWalking = Bool
- Sex = Catégorique
- AgeCategory = Catégorique
- Race = Catégorique
- Diabetic = Bool
- PhysicalActivity = Bool
- GenHealth = Catégorique
- SleepTime = Float
- Asthme = Bool
- KidneyDisease = Bool
- SkinCancer = Bool


## Vérification des premières lignes de données

In [5]:
pd_Data.head

<bound method NDFrame.head of        HeartDisease    BMI Smoking AlcoholDrinking Stroke  PhysicalHealth  \
0                No  16.60     Yes              No     No             3.0   
1                No  20.34      No              No    Yes             0.0   
2                No  26.58     Yes              No     No            20.0   
3                No  24.21      No              No     No             0.0   
4                No  23.71      No              No     No            28.0   
...             ...    ...     ...             ...    ...             ...   
319790          Yes  27.41     Yes              No     No             7.0   
319791           No  29.84     Yes              No     No             0.0   
319792           No  24.24      No              No     No             0.0   
319793           No  32.81      No              No     No             0.0   
319794           No  46.56      No              No     No             0.0   

        MentalHealth DiffWalking     Sex  Age

## Le fichier était déjà "cleaned", donc j'ai créé une version avec des données manquantes pour tester la version finale


In [6]:
pd_Data.isnull().sum()

HeartDisease        0
BMI                 1
Smoking             0
AlcoholDrinking     1
Stroke              0
PhysicalHealth      2
MentalHealth        0
DiffWalking         0
Sex                 1
AgeCategory         1
Race                1
Diabetic            1
PhysicalActivity    0
GenHealth           1
SleepTime           1
Asthma              0
KidneyDisease       0
SkinCancer          0
dtype: int64

In [7]:
pd_Data = pd_Data.dropna(how='any', axis=0)
pd_Data.isnull().sum()

HeartDisease        0
BMI                 0
Smoking             0
AlcoholDrinking     0
Stroke              0
PhysicalHealth      0
MentalHealth        0
DiffWalking         0
Sex                 0
AgeCategory         0
Race                0
Diabetic            0
PhysicalActivity    0
GenHealth           0
SleepTime           0
Asthma              0
KidneyDisease       0
SkinCancer          0
dtype: int64

## Transformation des données
- Changer les Bool, Yes/No par 1/0
- Encoding les catégorique : Sex, AgeCategory, Race, GenHealth
- Normaliser les Floats : BMI, PhysicalHealth, MentalHealth, SleepTime

# Transformer les Variables booléenne de "Yes/No" à 1 et 0.

In [8]:
V_bool = ['HeartDisease', 'Smoking', 'AlcoholDrinking', 'Stroke', 'DiffWalking', 'Diabetic', 'PhysicalActivity', 'Asthma', 'KidneyDisease', 'SkinCancer']

for cat in V_bool:
    pd_Data[cat] = np.where(pd_Data[cat] == 'Yes', 1, 0)


C:\Users\Formation RS\AppData\Local\Temp\ipykernel_16876\1350661429.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_Data[cat] = np.where(pd_Data[cat] == 'Yes', 1, 0)


# Normaliser les variable numériques

In [9]:
for df in ['BMI', 'PhysicalHealth', 'MentalHealth', 'SleepTime']:
    pd_Data[df] = (pd_Data[df] - pd_Data[df].min()) / (pd_Data[df].max() - pd_Data[df].min())

C:\Users\Formation RS\AppData\Local\Temp\ipykernel_16876\2256579064.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_Data[df] = (pd_Data[df] - pd_Data[df].min()) / (pd_Data[df].max() - pd_Data[df].min())


# Encoder les variables Catégoriques

In [10]:
# J'ai essayé de les faire d'un coup avec une liste, mais sans succès; donc je les fait une par une.
pd_Data_Encoded = pd.get_dummies(pd_Data, columns=['Sex'], drop_first=True)
pd_Data_Encoded = pd.get_dummies(pd_Data_Encoded, columns=['AgeCategory'])
pd_Data_Encoded = pd.get_dummies(pd_Data_Encoded, columns=['GenHealth'])
pd_Data_Encoded = pd.get_dummies(pd_Data_Encoded, columns=['Race'])

## Validation que les données sont bien transformées

In [11]:
print(pd_Data_Encoded.columns)
print(pd_Data_Encoded.head)

Index(['HeartDisease', 'BMI', 'Smoking', 'AlcoholDrinking', 'Stroke',
       'PhysicalHealth', 'MentalHealth', 'DiffWalking', 'Diabetic',
       'PhysicalActivity', 'SleepTime', 'Asthma', 'KidneyDisease',
       'SkinCancer', 'Sex_Male', 'AgeCategory_18-24', 'AgeCategory_25-29',
       'AgeCategory_30-34', 'AgeCategory_35-39', 'AgeCategory_40-44',
       'AgeCategory_45-49', 'AgeCategory_50-54', 'AgeCategory_55-59',
       'AgeCategory_60-64', 'AgeCategory_65-69', 'AgeCategory_70-74',
       'AgeCategory_75-79', 'AgeCategory_80 or older', 'GenHealth_Excellent',
       'GenHealth_Fair', 'GenHealth_Good', 'GenHealth_Poor',
       'GenHealth_Very good', 'Race_American Indian/Alaskan Native',
       'Race_Asian', 'Race_Black', 'Race_Hispanic', 'Race_Other',
       'Race_White'],
      dtype='object')
<bound method NDFrame.head of         HeartDisease       BMI  Smoking  AlcoholDrinking  Stroke  \
0                  0  0.055294        1                0       0   
1                  0  0.10

# Séparer les données en Train et Test et les étiquettes

In [12]:

X_train, X_test, y_train, y_test = train_test_split(
    pd_Data_Encoded.drop(['HeartDisease'], axis=1),
    pd_Data_Encoded['HeartDisease'], # the target
    test_size = 0.2,
    random_state=42)

# sauvegarder les données transformés
X_train.to_csv("X_train.csv")
X_test.to_csv("X_test.csv")
y_train.to_csv("y_train.csv")
y_test.to_csv("y_test.csv")

# Validation de la séparation.
X_train.shape, X_test.shape

((255828, 38), (63957, 38))

# Entrainer le model

In [ ]:
HD_model = LogisticRegression()

HD_model.fit(X_train, y_train)

# Prédire avec les données Test

In [ ]:
y_pred = HD_model.predict(X_test)

## Validation des résultats

### Micro-average: Calculates metrics globally by counting the total true positives, false negatives and false positives.
### Macro-average: Averages the F1 score for each class without considering class imbalance.
### Weighted-average: Considers class imbalance by weighting the F1 scores by the number of true instances for each class.

#### https://www.geeksforgeeks.org/f1-score-in-machine-learning/

In [20]:
print(f'Accuracy_score: {accuracy_score(y_test,y_pred)}')
print(f'Precission_score: {precision_score(y_test,y_pred)}')
print(f'Recall_score: {recall_score(y_test,y_pred)}')
print(f'F1-score par class: {f1_score(y_test,y_pred, average=None)}')
print(f'F1-score micro moyenne: {f1_score(y_test,y_pred, average='micro')}')
print(f'F1-score macro moyenne: {f1_score(y_test,y_pred, average='macro')}')
print(f'F1-score mpyenne par poid: {f1_score(y_test,y_pred, average='weighted')}')

Accuracy_score: 0.9151148427849962
Precission_score: 0.5617224880382775
Recall_score: 0.10561353004677941
F1-score par class: [0.95524726 0.17779797]
F1-score micro moyenne: 0.9151148427849962
F1-score macro moyenne: 0.5665226138347385
F1-score mpyenne par poid: 0.8876852523596988


# Sauvegarder le model

In [15]:
pickle.dump(HD_model, open("model.h5", "wb"))

## J'ai voulu tester comment faire afficher le output.

In [ ]:
testsum = y_pred.sum()
print(testsum)
#for test in y_pred:
#    if test:
#        print(test)

df_Pred_Result = X_test
df_Pred_Result['Heart_Disease'] = y_pred

df_Pred_Result.to_csv("HD_Test_Result.csv")

df_Pred_Result.head()

1045


,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Diabetic,PhysicalActivity,SleepTime,...,GenHealth_Good,GenHealth_Poor,GenHealth_Very good,Race_American Indian/Alaskan Native,Race_Asian,Race_Black,Race_Hispanic,Race_Other,Race_White,Heart_Disease
164503,0.138839,1,0,0,0.133333,0.333333,0,0,1,0.173913,...,True,False,False,False,True,False,False,False,False,0
232948,0.164916,0,0,0,0.000000,0.000000,0,0,1,0.217391,...,False,False,True,False,False,False,False,False,True,0
299931,0.159000,0,0,0,0.000000,0.000000,1,1,0,0.304348,...,True,False,False,False,False,False,True,False,False,0
267388,0.171315,1,0,0,0.000000,0.000000,0,0,0,0.304348,...,True,False,False,False,False,False,True,False,False,0
92406,0.218399,0,0,0,0.233333,0.166667,0,0,1,0.260870,...,False,False,True,False,False,False,False,False,True,0


# Dans un pipeline, je ne crois pas que l'on peut y ajouter un output comme des graphiques.
# Par contre, dans un fichier Excel, on pourrait créer un visuel des valeures positives et même ajouter des graphiques pour les utilisateurs.